1. Phillips’ goal is to measure the incremental effect of local search advertising on
Google Maps, using the data described in Exhibit 3. What kind of modeling approach
would be best suited for this task, and why?
2. Conduct the analysis using the modeling approach you described in your answer to
question 1. What is the estimated effect?
3. Based on your analysis, should Starbucks implement more local search advertising
on Google Maps in the future? First provide a yes/no answer, and then explain your
reasoning.
4. If Phillips has the budget to run another test next year, should he set up the test
differently? If so, how?

In [4]:
import pandas as pd
df = pd.read_csv('starbucks.csv')
df

,geo_ID,geo_name,month,sales,price,treatment,age_21over_pct,age_55over_pct,age_85over_pct,race_native_pct,race_asian_pct,race_black_pct,race_hispanic_pct,race_white_pct,unemployed_pct,incomeK
0,1,Portland ME,1,80.495,2.9269,1,72.61,25.06,1.05,0.57,6.35,6.31,3.79,84.78,5.64,40.88
1,1,Portland ME,2,106.500,2.7980,1,72.47,25.10,0.99,0.48,6.76,6.90,3.78,85.67,5.54,40.15
2,1,Portland ME,3,78.382,3.0511,1,73.01,25.13,1.15,0.60,7.01,6.28,3.48,83.55,5.58,39.93
3,1,Portland ME,4,110.250,3.0942,1,72.41,25.02,0.98,0.47,6.52,6.70,4.40,83.60,5.24,40.23
4,1,Portland ME,5,135.150,2.9836,1,72.43,25.21,0.98,0.50,6.39,6.07,3.65,84.75,5.24,41.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2395,200,Seattle,8,152.240,4.7111,0,73.69,27.15,2.89,0.79,2.99,3.53,3.60,89.29,4.60,42.26
2396,200,Seattle,9,130.320,4.6967,0,73.46,26.57,2.91,0.72,4.59,4.36,4.02,88.81,4.74,41.77
2397,200,Seattle,10,132.290,4.8923,0,73.78,26.70,2.95,0.79,4.45,2.78,5.20,89.74,4.73,42.31
2398,200,Seattle,11,134.570,4.7589,0,73.76,26.74,2.92,0.75,3.54,4.33,5.82,89.78,4.92,42.13


##1. Solution is Difference-in-Differences regression with market and month fixed effects. Since we observe multiple geographic markets over time and some are assigned to treatment, DiD allows us to compare how sales change in treated markets relative to control markets.

This approach controls for time-invariant differences across markets (like demographics or baseline demand) and common time shocks (like seasonality). By including additional controls such as price and income, we further reduce confounding.

Under the parallel trends assumption, the treatment coefficient gives the incremental lift in sales attributable to the advertising.

In [5]:
df["post"] = (df["month"] >= 6).astype(int)

model = smf.ols(
    "sales ~ treatment*post + price + incomeK + unemployed_pct \
     + C(geo_ID) + C(month)",
    data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["geo_ID"]})

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.294
Model:                            OLS   Adj. R-squared:                  0.225
Method:                 Least Squares   F-statistic:                     29.68
Date:                Thu, 26 Feb 2026   Prob (F-statistic):           1.07e-42
Time:                        07:24:37   Log-Likelihood:                -10484.
No. Observations:                2400   AIC:                         2.140e+04
Df Residuals:                    2185   BIC:                         2.264e+04
Df Model:                         214                                         
Covariance Type:              cluster                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept          242.0769     32.984  

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 216, but rank is 15
  warnings.warn('covariance of constraints does not have full '


##3.
YES.
 In Difference-in-Differences model, the key coefficient is the interaction treatment:post, which estimates the incremental lift from ads during June–December in treated markets relative to control markets. That coefficient is +9.36 and statistically significant (p < 0.001, CI roughly [5.84, 12.87]), indicating the ads increased average unit sales by about 9 units per hour per store (given your sales definition). Price is significantly negative (as expected), while income and unemployment are not significant here. The negative main treatment term just reflects baseline differences between treated and control markets before the campaign, and the positive post term reflects overall seasonality/time effects neither contradicts the incremental lift.

##4.
Yes. We should redesign it slightly to improve learning and robustness.

First, he should stratify randomization (e.g., by baseline sales or region) to ensure better balance between treatment and control markets. Second, he should incorporate an event-study approach to test parallel trends and observe how effects evolve over time, rather than relying on a single post-period estimate. Finally, if feasible, testing multiple keywords or budget levels would help optimize ROI rather than just measure average lift.